# Step 2E — Generate Retail Sales Transactions

## 1. Project Configuration

## 2. Load Dimension Tables

## 3. Validate Dimension Tables

## 4. Generate Transaction Skeleton

## 5. Generate Transaction Dates

## 6. Assign Stores

## 7. Assign Customers

## 8. Assign Products

## 9. Generate Quantities

## 10. Generate Pricing

## 11. Generate Discounts

## 12. Calculate Financial Metrics

## 13. Assign Payment Methods

## 14. Assign Sales Channels

## 15. Build Final Fact Table

## 16. Data Quality Validation

## 17. Business Sanity Checks

## 18. Save Dataset

In [188]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

N_TRANSACTIONS = 120_000

In [3]:
N_TRANSACTIONS

120000

### Load Our Dimension Tables

In [189]:
data_path = Path("../data/raw")

df_store = pd.read_csv(
    data_path / "dim_store.csv"
)

df_product = pd.read_csv(
    data_path / "dim_product.csv"
)

df_customer = pd.read_csv(
    data_path / "dim_customer.csv"
)

df_date = pd.read_csv(
    data_path / "dim_date.csv"
)

### Check

In [190]:
df_store.shape

(25, 9)

In [6]:
df_product.shape

(500, 7)

In [7]:
df_customer.shape

(5000, 5)

In [8]:
df_date.shape

(731, 12)

### Validate Before Generating Sales

In [9]:
assert df_store["store_id"].is_unique
assert df_product["product_id"].is_unique
assert df_customer["customer_id"].is_unique
assert df_date["date"].is_unique

## 4. Generate Transaction Skeleton

In [10]:
N_TRANSACTIONS = 120_000

transaction_ids = np.arange(1, N_TRANSACTIONS + 1)

df_sales = pd.DataFrame({
    "transaction_id": [
        f"T{i:06d}" for i in transaction_ids
    ]
})

df_sales.head()

,transaction_id
0,T000001
1,T000002
2,T000003
3,T000004
4,T000005


In [11]:
df_sales.shape

(120000, 1)

In [12]:
df_sales["transaction_id"].is_unique

True

## 5. Generate Transaction Dates

In [13]:
df_date["date"] = pd.to_datetime(df_date["date"])

In [14]:
df_date[["date"]].head()

,date
0,2024-01-01
1,2024-01-02
2,2024-01-03
3,2024-01-04
4,2024-01-05


In [15]:
df_date["date"].min(), df_date["date"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

### Create Day-of-Week Behavior

In [16]:
date_weights = df_date[["date"]].copy()

date_weights["day_of_week"] = date_weights["date"].dt.dayofweek

In [17]:
weekday_weights = {
    0: 1.00,  # Monday
    1: 1.00,  # Tuesday
    2: 1.00,  # Wednesday
    3: 1.05,  # Thursday
    4: 1.20,  # Friday
    5: 1.30,  # Saturday
    6: 1.10   # Sunday
}

In [18]:
date_weights["weekday_weight"] = (
    date_weights["day_of_week"]
    .map(weekday_weights)
)

In [19]:
date_weights.head(10)

,date,day_of_week,weekday_weight
0,2024-01-01,0,1.00
1,2024-01-02,1,1.00
2,2024-01-03,2,1.00
3,2024-01-04,3,1.05
4,2024-01-05,4,1.20
5,2024-01-06,5,1.30
6,2024-01-07,6,1.10
7,2024-01-08,0,1.00
8,2024-01-09,1,1.00
9,2024-01-10,2,1.00


### Add Monthly Seasonality

In [20]:
date_weights["month"] = date_weights["date"].dt.month

In [21]:
monthly_weights = {
    1: 0.95,
    2: 0.95,
    3: 1.05,
    4: 1.00,
    5: 1.00,
    6: 1.05,
    7: 1.00,
    8: 1.00,
    9: 1.05,
    10: 1.10,
    11: 1.15,
    12: 1.20
}

In [22]:
date_weights["seasonal_weight"] = (
    date_weights["month"].map(monthly_weights)
)

### Add 2025 Growth

In [23]:
date_weights["year"] = date_weights["date"].dt.year

date_weights["year_weight"] = np.where(
    date_weights["year"] == 2025,
    1.08,
    1.00
)

### Combine the Weights

In [24]:
date_weights["weight"] = (
    date_weights["weekday_weight"]
    * date_weights["seasonal_weight"]
    * date_weights["year_weight"]
)

In [25]:
date_weights[
    [
        "date",
        "weekday_weight",
        "seasonal_weight",
        "year_weight",
        "weight"
    ]
].head(10)

,date,weekday_weight,seasonal_weight,year_weight,weight
0,2024-01-01,1.00,0.95,1.0,0.9500
1,2024-01-02,1.00,0.95,1.0,0.9500
2,2024-01-03,1.00,0.95,1.0,0.9500
3,2024-01-04,1.05,0.95,1.0,0.9975
4,2024-01-05,1.20,0.95,1.0,1.1400
5,2024-01-06,1.30,0.95,1.0,1.2350
6,2024-01-07,1.10,0.95,1.0,1.0450
7,2024-01-08,1.00,0.95,1.0,0.9500
8,2024-01-09,1.00,0.95,1.0,0.9500
9,2024-01-10,1.00,0.95,1.0,0.9500


### Convert Weights to Probabilities

In [26]:
date_weights["probability"] = (
    date_weights["weight"]
    / date_weights["weight"].sum()
)

In [27]:
date_weights["probability"].sum()

np.float64(1.0)

In [28]:
np.isclose(
    date_weights["probability"].sum(),
    1.0
)

np.True_

### Generate 120,000 Dates

In [29]:
df_sales["transaction_date"] = np.random.choice(
    date_weights["date"],
    size=N_TRANSACTIONS,
    p=date_weights["probability"]
)

In [30]:
df_sales.head()

,transaction_id,transaction_date
0,T000001,2024-10-20
1,T000002,2025-12-01
2,T000003,2025-07-03
3,T000004,2025-03-29
4,T000005,2024-05-04


### Validate the Dates

In [31]:
df_sales["transaction_date"].min()
df_sales["transaction_date"].max()

Timestamp('2025-12-31 00:00:00')

In [32]:
df_sales["transaction_date"].isna().sum()

np.int64(0)

In [33]:
df_sales.shape

(120000, 2)

### Inspect Daily Distribution

In [34]:
daily_sales_lines = (
    df_sales
    .groupby("transaction_date")
    .size()
    .reset_index(name="transaction_lines")
)

In [35]:
daily_sales_lines.head()

,transaction_date,transaction_lines
0,2024-01-01,133
1,2024-01-02,129
2,2024-01-03,131
3,2024-01-04,153
4,2024-01-05,141


In [36]:
daily_sales_lines["transaction_lines"].describe()

count    731.000000
mean     164.158687
std       23.903619
min      105.000000
25%      146.000000
50%      163.000000
75%      179.000000
max      256.000000
Name: transaction_lines, dtype: float64

### Check Monthly Distribution

In [37]:
monthly_distribution = (
    df_sales
    .assign(
        year=df_sales["transaction_date"].dt.year,
        month=df_sales["transaction_date"].dt.month
    )
    .groupby(["year", "month"])
    .size()
    .reset_index(name="transaction_lines")
)

In [38]:
monthly_distribution

,year,month,transaction_lines
0,2024,1,4365
1,2024,2,4276
2,2024,3,5012
3,2024,4,4494
4,2024,5,4702
5,2024,6,4816
6,2024,7,4642
7,2024,8,4640
8,2024,9,4942
9,2024,10,4924


### Compare 2024 vs 2025

In [39]:
year_distribution = (
    df_sales["transaction_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

year_distribution

transaction_date
2024    57675
2025    62325
Name: count, dtype: int64

### Check Day-of-Week Distribution

In [40]:
dow_distribution = (
    df_sales["transaction_date"]
    .dt.day_name()
    .value_counts()
)

In [41]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_distribution = dow_distribution.reindex(day_order)

dow_distribution

transaction_date
Monday       15808
Tuesday      15863
Wednesday    15938
Thursday     16262
Friday       18679
Saturday     20256
Sunday       17194
Name: count, dtype: int64

In [42]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_distribution = dow_distribution.reindex(day_order)

dow_distribution

transaction_date
Monday       15808
Tuesday      15863
Wednesday    15938
Thursday     16262
Friday       18679
Saturday     20256
Sunday       17194
Name: count, dtype: int64

### Assign Stores

In [43]:
df_store.head()

,store_id,store_name,city,region,store_type,store_size_sqft,opening_date,target_sales,target_margin
0,GM001,GulfMart Riyadh North,Riyadh,Central,Hypermarket,30251,2020-02-08,13023504,0.28
1,GM002,GulfMart Riyadh South,Riyadh,Central,Hypermarket,34235,2020-04-22,12605221,0.28
2,GM003,GulfMart Riyadh Central,Riyadh,Central,Supermarket,13929,2021-05-17,9748688,0.30
3,GM004,GulfMart Riyadh East,Riyadh,Central,Supermarket,13013,2020-10-02,9920367,0.30
4,GM005,GulfMart Riyadh West,Riyadh,Central,Supermarket,12226,2019-12-11,8949709,0.30


In [44]:
df_store.columns.tolist()

['store_id',
 'store_name',
 'city',
 'region',
 'store_type',
 'store_size_sqft',
 'opening_date',
 'target_sales',
 'target_margin']

In [45]:
df_store[
    [
        "store_id",
        "store_name",
        "city",
        "region",
        "store_type",
        "store_size_sqft",
        "target_sales"
    ]
].head(10)

,store_id,store_name,city,region,store_type,store_size_sqft,target_sales
0,GM001,GulfMart Riyadh North,Riyadh,Central,Hypermarket,30251,13023504
1,GM002,GulfMart Riyadh South,Riyadh,Central,Hypermarket,34235,12605221
2,GM003,GulfMart Riyadh Central,Riyadh,Central,Supermarket,13929,9748688
3,GM004,GulfMart Riyadh East,Riyadh,Central,Supermarket,13013,9920367
4,GM005,GulfMart Riyadh West,Riyadh,Central,Supermarket,12226,8949709
5,GM006,GulfMart Riyadh Express 1,Riyadh,Central,Express,7225,2879653
6,GM007,GulfMart Riyadh Express 2,Riyadh,Central,Express,5828,3610178
7,GM008,GulfMart Jeddah North,Jeddah,Western,Hypermarket,32909,12300269
8,GM009,GulfMart Jeddah South,Jeddah,Western,Hypermarket,30588,14431373
9,GM010,GulfMart Jeddah Central,Jeddah,Western,Supermarket,14023,6566510


In [46]:
df_store["store_type"].value_counts()

store_type
Supermarket     10
Express          7
Hypermarket      5
Neighborhood     3
Name: count, dtype: int64

In [47]:
df_store["city"].value_counts()

city
Riyadh     7
Jeddah     5
Dammam     3
Makkah     3
Khobar     2
Madinah    2
Abha       2
Tabuk      1
Name: count, dtype: int64

### Create the store-type weights

In [48]:
store_type_weights = {
    "Hypermarket": 1.50,
    "Supermarket": 1.20,
    "Express": 0.80,
    "Neighborhood": 0.70
}

### Map the weights to your stores

In [49]:
df_store["store_type_weight"] = (
    df_store["store_type"]
    .map(store_type_weights)
)

In [50]:
df_store[
    [
        "store_id",
        "store_type",
        "store_type_weight"
    ]
]

,store_id,store_type,store_type_weight
0,GM001,Hypermarket,1.5
1,GM002,Hypermarket,1.5
2,GM003,Supermarket,1.2
3,GM004,Supermarket,1.2
4,GM005,Supermarket,1.2
5,GM006,Express,0.8
6,GM007,Express,0.8
7,GM008,Hypermarket,1.5
8,GM009,Hypermarket,1.5
9,GM010,Supermarket,1.2


In [51]:
print(
    "Missing store type weights:",
    df_store["store_type_weight"].isna().sum()
)

Missing store type weights: 0


### Store Size Weight

In [52]:
df_store["size_weight"] = (
    df_store["store_size_sqft"]
    / df_store["store_size_sqft"].median()
)

In [53]:
df_store[
    [
        "store_id",
        "store_size_sqft",
        "size_weight"
    ]
].head()

,store_id,store_size_sqft,size_weight
0,GM001,30251,2.312768
1,GM002,34235,2.617355
2,GM003,13929,1.064908
3,GM004,13013,0.994878
4,GM005,12226,0.934709


### Create Store Performance Factor

In [54]:
df_store["performance_factor"] = np.random.uniform(
    0.85,
    1.20,
    size=len(df_store)
).round(3)

In [55]:
df_store[
    [
        "store_id",
        "store_name",
        "store_type",
        "store_size_sqft",
        "performance_factor"
    ]
]

,store_id,store_name,store_type,store_size_sqft,performance_factor
0,GM001,GulfMart Riyadh North,Hypermarket,30251,0.895
1,GM002,GulfMart Riyadh South,Hypermarket,34235,1.039
2,GM003,GulfMart Riyadh Central,Supermarket,13929,1.065
3,GM004,GulfMart Riyadh East,Supermarket,13013,1.028
4,GM005,GulfMart Riyadh West,Supermarket,12226,1.031
5,GM006,GulfMart Riyadh Express 1,Express,7225,0.907
6,GM007,GulfMart Riyadh Express 2,Express,5828,1.060
7,GM008,GulfMart Jeddah North,Hypermarket,32909,1.192
8,GM009,GulfMart Jeddah South,Hypermarket,30588,1.192
9,GM010,GulfMart Jeddah Central,Supermarket,14023,0.972


In [56]:
df_store["performance_factor"].describe()

count    25.000000
mean      1.020320
std       0.101259
min       0.868000
25%       0.911000
50%       1.028000
75%       1.105000
max       1.192000
Name: performance_factor, dtype: float64

In [57]:
print(
    "Missing performance factors:",
    df_store["performance_factor"].isna().sum()
)

Missing performance factors: 0


### Create the Final Store Weight

In [58]:
df_store["transaction_weight"] = (
    df_store["store_type_weight"]
    * df_store["size_weight"]
    * df_store["performance_factor"]
)

In [59]:
df_store[
    [
        "store_id",
        "store_type",
        "store_size_sqft",
        "store_type_weight",
        "size_weight",
        "performance_factor",
        "transaction_weight"
    ]
].sort_values(
    "transaction_weight",
    ascending=False
)

,store_id,store_type,store_size_sqft,store_type_weight,size_weight,performance_factor,transaction_weight
7,GM008,Hypermarket,32909,1.5,2.515979,1.192,4.498570
8,GM009,Hypermarket,30588,1.5,2.338532,1.192,4.181295
1,GM002,Hypermarket,34235,1.5,2.617355,1.039,4.079147
12,GM013,Hypermarket,39537,1.5,3.022706,0.898,4.071586
0,GM001,Hypermarket,30251,1.5,2.312768,0.895,3.104890
13,GM014,Supermarket,17891,1.2,1.367813,1.109,1.820286
22,GM023,Supermarket,16431,1.2,1.256193,1.160,1.748620
10,GM011,Supermarket,16966,1.2,1.297095,1.105,1.719948
20,GM021,Supermarket,15586,1.2,1.191590,0.978,1.398450
2,GM003,Supermarket,13929,1.2,1.064908,1.065,1.360953


### Validate transaction weights

In [60]:
print(
    "Missing transaction weights:",
    df_store["transaction_weight"].isna().sum()
)

print(
    "Negative transaction weights:",
    (df_store["transaction_weight"] < 0).sum()
)

Missing transaction weights: 0
Negative transaction weights: 0


### Convert weight to prababilities

In [61]:
df_store["store_probability"] = (
    df_store["transaction_weight"]
    / df_store["transaction_weight"].sum()
)

In [62]:
print(
    "Missing probabilities:",
    df_store["store_probability"].isna().sum()
)

print(
    "Probability total:",
    df_store["store_probability"].sum()
)

Missing probabilities: 0
Probability total: 0.9999999999999999


In [63]:
print(
    "Minimum probability:",
    df_store["store_probability"].min()
)

print(
    "Maximum probability:",
    df_store["store_probability"].max()
)

Minimum probability: 0.006540704716814694
Maximum probability: 0.11845190812993542


### Assign Stores to Transactions

In [64]:
df_sales["store_id"] = np.random.choice(
    df_store["store_id"],
    size=N_TRANSACTIONS,
    p=df_store["store_probability"]
)

In [65]:
df_sales.head()

,transaction_id,transaction_date,store_id
0,T000001,2024-10-20,GM007
1,T000002,2025-12-01,GM004
2,T000003,2025-07-03,GM013
3,T000004,2025-03-29,GM004
4,T000005,2024-05-04,GM012


### Validate Store IDs

In [66]:
df_sales["store_id"].isin(
    df_store["store_id"]
).all()

np.True_

### Missing Values

In [67]:
df_sales["store_id"].isna().sum()

np.int64(0)

### Store Transaction Distribution

In [68]:
store_transaction_distribution = (
    df_sales["store_id"]
    .value_counts()
    .rename_axis("store_id")
    .reset_index(name="transaction_lines")
)

In [69]:
store_transaction_distribution

,store_id,transaction_lines
0,GM008,14227
1,GM009,13223
2,GM013,12949
3,GM002,12816
4,GM001,9801
5,GM014,5698
6,GM023,5620
7,GM011,5462
8,GM021,4451
9,GM003,4325


### Compare Actual Distribution With Expected Probability

In [70]:
store_distribution_check = store_transaction_distribution.merge(
    df_store[
        [
            "store_id",
            "store_probability",
            "transaction_weight",
            "store_type",
            "store_size_sqft"
        ]
    ],
    on="store_id",
    how="left"
)

In [71]:
store_distribution_check["expected_lines"] = (
    store_distribution_check["store_probability"]
    * N_TRANSACTIONS
)

In [72]:
store_distribution_check[
    [
        "store_id",
        "store_type",
        "store_size_sqft",
        "transaction_lines",
        "expected_lines"
    ]
].sort_values(
    "transaction_lines",
    ascending=False
)

,store_id,store_type,store_size_sqft,transaction_lines,expected_lines
0,GM008,Hypermarket,32909,14227,14214.228976
1,GM009,Hypermarket,30588,13223,13211.730405
2,GM013,Hypermarket,39537,12949,12865.077758
3,GM002,Hypermarket,34235,12816,12888.970980
4,GM001,Hypermarket,30251,9801,9810.590243
5,GM014,Supermarket,17891,5698,5751.597880
6,GM023,Supermarket,16431,5620,5525.153371
7,GM011,Supermarket,16966,5462,5434.556319
8,GM021,Supermarket,15586,4451,4418.713867
9,GM003,Supermarket,13929,4325,4300.232125


Check Store Performance Potential

In [73]:
df_store[
    [
        "store_id",
        "store_name",
        "city",
        "region",
        "store_type",
        "store_size_sqft",
        "transaction_weight",
        "store_probability"
    ]
].sort_values(
    "transaction_weight",
    ascending=False
)

,store_id,store_name,city,region,store_type,store_size_sqft,transaction_weight,store_probability
7,GM008,GulfMart Jeddah North,Jeddah,Western,Hypermarket,32909,4.498570,0.118452
8,GM009,GulfMart Jeddah South,Jeddah,Western,Hypermarket,30588,4.181295,0.110098
1,GM002,GulfMart Riyadh South,Riyadh,Central,Hypermarket,34235,4.079147,0.107408
12,GM013,GulfMart Dammam North,Dammam,Eastern,Hypermarket,39537,4.071586,0.107209
0,GM001,GulfMart Riyadh North,Riyadh,Central,Hypermarket,30251,3.104890,0.081755
13,GM014,GulfMart Dammam Central,Dammam,Eastern,Supermarket,17891,1.820286,0.047930
22,GM023,GulfMart Abha Central,Abha,Southern,Supermarket,16431,1.748620,0.046043
10,GM011,GulfMart Jeddah East,Jeddah,Western,Supermarket,16966,1.719948,0.045288
20,GM021,GulfMart Madinah Central,Madinah,Western,Supermarket,15586,1.398450,0.036823
2,GM003,GulfMart Riyadh Central,Riyadh,Central,Supermarket,13929,1.360953,0.035835


In [74]:
df_sales.shape

(120000, 3)

In [75]:
df_sales["store_id"].isin(
    df_store["store_id"]
).all()

np.True_

In [76]:
df_sales["store_id"].isna().sum()

np.int64(0)

In [77]:
df_sales["store_id"].nunique()

25

In [78]:
store_transaction_distribution

,store_id,transaction_lines
0,GM008,14227
1,GM009,13223
2,GM013,12949
3,GM002,12816
4,GM001,9801
5,GM014,5698
6,GM023,5620
7,GM011,5462
8,GM021,4451
9,GM003,4325


## Assign Customers

In [79]:
df_customer.head()

,customer_id,gender,age,city,customer_segment
0,C00001,Male,31,Riyadh,New
1,C00002,Female,38,Dammam,Regular
2,C00003,Female,46,Khobar,Regular
3,C00004,Female,29,Jeddah,Regular
4,C00005,Male,47,Madinah,Regular


In [80]:
df_customer.columns.tolist()

['customer_id', 'gender', 'age', 'city', 'customer_segment']

In [81]:
df_customer.shape

(5000, 5)

In [82]:
df_customer["customer_segment"].value_counts()

customer_segment
Regular       2169
Occasional    1520
New            809
Premium        502
Name: count, dtype: int64

In [83]:
df_customer["city"].value_counts()

city
Riyadh     1446
Jeddah     1101
Dammam      600
Makkah      538
Khobar      412
Madinah     411
Abha        293
Tabuk       199
Name: count, dtype: int64

In [84]:
df_customer["customer_id"].is_unique

True

### Create Segment Weights

In [85]:
customer_segment_weights = {
    "Premium": 3.0,
    "Regular": 1.8,
    "Occasional": 0.8,
    "New": 0.5
}

In [86]:
df_customer["segment_weight"] = (
    df_customer["customer_segment"]
    .map(customer_segment_weights)
)

In [87]:
df_customer[
    [
        "customer_id",
        "customer_segment",
        "segment_weight"
    ]
].head(10)

,customer_id,customer_segment,segment_weight
0,C00001,New,0.5
1,C00002,Regular,1.8
2,C00003,Regular,1.8
3,C00004,Regular,1.8
4,C00005,Regular,1.8
5,C00006,Occasional,0.8
6,C00007,New,0.5
7,C00008,New,0.5
8,C00009,New,0.5
9,C00010,Regular,1.8


### Add Customer-Level Variation

In [88]:
customer_variation = np.random.lognormal(
    mean=0,
    sigma=0.25,
    size=len(df_customer)
)

In [89]:
df_customer["customer_variation"] = customer_variation

In [90]:
df_customer["purchase_weight"] = (
    df_customer["segment_weight"]
    * df_customer["customer_variation"]
)

In [91]:
df_customer[
    [
        "customer_id",
        "customer_segment",
        "segment_weight",
        "customer_variation",
        "purchase_weight"
    ]
].sort_values(
    "purchase_weight",
    ascending=False
).head(10)

,customer_id,customer_segment,segment_weight,customer_variation,purchase_weight
3212,C03213,Premium,3.0,2.162457,6.487372
1218,C01219,Premium,3.0,2.130256,6.390769
390,C00391,Premium,3.0,1.994325,5.982974
1778,C01779,Premium,3.0,1.969607,5.908822
3989,C03990,Premium,3.0,1.935143,5.805430
4261,C04262,Premium,3.0,1.853739,5.561216
4151,C04152,Premium,3.0,1.847214,5.541641
209,C00210,Premium,3.0,1.789030,5.367091
2100,C02101,Premium,3.0,1.778390,5.335170
1880,C01881,Premium,3.0,1.746607,5.239821


### Geographic Affinity

### Build a Store-City Lookup

In [92]:
store_city_map = (
    df_store[
        ["store_id", "city"]
    ]
    .drop_duplicates()
)

In [93]:
store_city_map.head()

,store_id,city
0,GM001,Riyadh
1,GM002,Riyadh
2,GM003,Riyadh
3,GM004,Riyadh
4,GM005,Riyadh


### Assign Customers by Geographic Pool

### Create a Store Region Weight

In [94]:
df_store[
    ["store_id", "city", "region"]
].head(20)

,store_id,city,region
0,GM001,Riyadh,Central
1,GM002,Riyadh,Central
2,GM003,Riyadh,Central
3,GM004,Riyadh,Central
4,GM005,Riyadh,Central
5,GM006,Riyadh,Central
6,GM007,Riyadh,Central
7,GM008,Jeddah,Western
8,GM009,Jeddah,Western
9,GM010,Jeddah,Western


In [95]:
set(df_customer["city"]) - set(df_store["city"])

set()

In [96]:
set(df_store["city"]) - set(df_customer["city"])

set()

### Prepare Customer Pools

In [97]:
customer_pools = {
    city: group["customer_id"].to_numpy()
    for city, group in df_customer.groupby("city")
}

In [98]:
customer_weight_pools = {
    city: group["purchase_weight"].to_numpy()
    for city, group in df_customer.groupby("city")
}

### Create a Customer Assignment Array

In [99]:
df_sales["customer_id"] = None

In [100]:
store_city_lookup = (
    df_store
    .set_index("store_id")["city"]
    .to_dict()
)

In [101]:
df_sales["store_city"] = (
    df_sales["store_id"]
    .map(store_city_lookup)
)

In [102]:
df_sales[
    ["transaction_id", "store_id", "store_city"]
].head()

,transaction_id,store_id,store_city
0,T000001,GM007,Riyadh
1,T000002,GM004,Riyadh
2,T000003,GM013,Dammam
3,T000004,GM004,Riyadh
4,T000005,GM012,Jeddah


### Assign Customers by City

In [103]:
for city in df_sales["store_city"].dropna().unique():

    mask = df_sales["store_city"] == city
    n = mask.sum()

    customers = customer_pools[city]
    weights = customer_weight_pools[city]

    probabilities = weights / weights.sum()

    df_sales.loc[mask, "customer_id"] = np.random.choice(
        customers,
        size=n,
        p=probabilities
    )

### Validate Customer Assignment

In [104]:
df_sales["customer_id"].isna().sum()

np.int64(0)

In [105]:
df_sales["customer_id"].isin(
    df_customer["customer_id"]
).all()

np.True_

In [106]:
df_sales["customer_id"].nunique()

4982

### Check Customer Purchase Frequency

In [107]:
customer_frequency = (
    df_sales["customer_id"]
    .value_counts()
    .rename_axis("customer_id")
    .reset_index(name="transaction_lines")
)

In [108]:
customer_frequency["transaction_lines"].describe()

count    4982.000000
mean       24.086712
std        18.782938
min         1.000000
25%        10.000000
50%        19.000000
75%        34.000000
max       159.000000
Name: transaction_lines, dtype: float64

### Compare Frequency by Segment

In [109]:
customer_frequency = customer_frequency.merge(
    df_customer[
        [
            "customer_id",
            "customer_segment"
        ]
    ],
    on="customer_id",
    how="left"
)

In [110]:
customer_frequency.groupby(
    "customer_segment"
)["transaction_lines"].describe()

,count,mean,std,min,25%,50%,75%,max
customer_segment,,,,,,,,
New,795.0,8.427673,4.812948,1.0,5.0,8.0,12.0,25.0
Occasional,1516.0,13.498021,7.039399,1.0,8.0,13.0,18.0,47.0
Premium,502.0,52.109562,25.277343,2.0,34.0,51.5,69.0,159.0
Regular,2169.0,30.741355,15.011341,1.0,19.0,29.0,41.0,101.0


### Check Geographic Affinity

In [111]:
customer_store_city_check = df_sales.merge(
    df_customer[
        ["customer_id", "city"]
    ],
    on="customer_id",
    how="left"
)

In [112]:
(
    customer_store_city_check["store_city"]
    ==
    customer_store_city_check["city"]
).mean()

np.float64(1.0)

In [113]:
df_sales.head()

,transaction_id,transaction_date,store_id,customer_id,store_city
0,T000001,2024-10-20,GM007,C03392,Riyadh
1,T000002,2025-12-01,GM004,C02634,Riyadh
2,T000003,2025-07-03,GM013,C01832,Dammam
3,T000004,2025-03-29,GM004,C02598,Riyadh
4,T000005,2024-05-04,GM012,C01959,Jeddah


## Assigns Products

### Category Demand Weight

In [114]:
category_weights = {
    "Food & Beverages": 1.30,
    "Fresh Food": 1.25,
    "Health & Wellness": 1.05,
    "Personal Care": 0.95,
    "Beauty": 0.90,
    "Household": 0.90,
    "Baby Care": 0.85,
    "Electronics": 0.75
}

In [115]:
df_product["category"].unique()

<StringArray>
[        'Baby Care',            'Beauty',     'Personal Care',
       'Electronics',  'Food & Beverages',        'Fresh Food',
 'Health & Wellness',         'Household']
Length: 8, dtype: str

### Create the Category Weight

In [116]:
df_product["category_weight"] = (
    df_product["category"]
    .map(category_weights)
)

In [117]:
print(
    "Missing category weights:",
    df_product["category_weight"].isna().sum()
)

Missing category weights: 0


In [118]:
df_product[
    [
        "product_id",
        "product_name",
        "category",
        "category_weight"
    ]
].head(20)

,product_id,product_name,category,category_weight
0,P0001,Northern Star Baby Care Item 001,Baby Care,0.85
1,P0002,Rimal Beauty Item 002,Beauty,0.90
2,P0003,Nakhla Personal Care Item 003,Personal Care,0.95
3,P0004,Rimal Electronics Item 004,Electronics,0.75
4,P0005,Arabian Select Baby Care Item 005,Baby Care,0.85
5,P0006,Gulf Fresh Food & Beverages Item 006,Food & Beverages,1.30
6,P0007,Sahara Home Fresh Food Item 007,Fresh Food,1.25
7,P0008,Arabian Select Electronics Item 008,Electronics,0.75
8,P0009,Golden Basket Fresh Food Item 009,Fresh Food,1.25
9,P0010,Golden Basket Baby Care Item 010,Baby Care,0.85


In [119]:
df_product.groupby(
    "category"
)["category_weight"].agg(
    ["count", "min", "max"]
)

,count,min,max
category,,,
Baby Care,34,0.85,0.85
Beauty,49,0.90,0.90
Electronics,42,0.75,0.75
Food & Beverages,122,1.30,1.30
Fresh Food,70,1.25,1.25
Health & Wellness,56,1.05,1.05
Household,71,0.90,0.90
Personal Care,56,0.95,0.95


In [120]:
df_product["category"].unique()

<StringArray>
[        'Baby Care',            'Beauty',     'Personal Care',
       'Electronics',  'Food & Beverages',        'Fresh Food',
 'Health & Wellness',         'Household']
Length: 8, dtype: str

In [121]:
df_product["category_weight"].isna().sum()

np.int64(0)

### Product-Level Popularity

In [122]:
product_variation = np.random.lognormal(
    mean=0,
    sigma=0.45,
    size=len(df_product)
)

In [123]:
df_product["product_variation"] = product_variation

### Final product demand weight

In [124]:
df_product["product_demand_weight"] = (
    df_product["category_weight"]
    * df_product["product_variation"]
)

### Inspect the highest-demand products

In [125]:
df_product[
    [
        "product_id",
        "product_name",
        "category",
        "product_demand_weight"
    ]
].sort_values(
    "product_demand_weight",
    ascending=False
).head(15)

,product_id,product_name,category,product_demand_weight
417,P0418,Golden Basket Fresh Food Item 418,Fresh Food,4.381254
405,P0406,Al Waha Fresh Food Item 406,Fresh Food,3.849346
313,P0314,Eastern Choice Fresh Food Item 314,Fresh Food,3.817160
314,P0315,Eastern Choice Fresh Food Item 315,Fresh Food,3.753470
307,P0308,Sahara Home Fresh Food Item 308,Fresh Food,3.648873
448,P0449,Sahara Home Fresh Food Item 449,Fresh Food,3.555119
213,P0214,Rimal Household Item 214,Household,3.432061
497,P0498,Arabian Select Food & Beverages Item 498,Food & Beverages,3.415886
260,P0261,Palm Valley Food & Beverages Item 261,Food & Beverages,3.275735
258,P0259,Sahara Home Fresh Food Item 259,Fresh Food,3.131266


In [126]:
df_product[
    [
        "product_id",
        "product_name",
        "category",
        "product_demand_weight"
    ]
].sort_values(
    "product_demand_weight"
).head(15)

,product_id,product_name,category,product_demand_weight
305,P0306,Palm Valley Baby Care Item 306,Baby Care,0.264199
396,P0397,Palm Valley Personal Care Item 397,Personal Care,0.270964
241,P0242,Desert Harvest Personal Care Item 242,Personal Care,0.286034
257,P0258,Nakhla Beauty Item 258,Beauty,0.307962
469,P0470,Nakhla Beauty Item 470,Beauty,0.316143
80,P0081,Gulf Fresh Household Item 081,Household,0.319415
169,P0170,Palm Valley Personal Care Item 170,Personal Care,0.336218
58,P0059,Sahara Home Electronics Item 059,Electronics,0.337159
423,P0424,Al Waha Electronics Item 424,Electronics,0.339095
101,P0102,Gulf Fresh Personal Care Item 102,Personal Care,0.355999


### Create Store Assortment Probability

In [127]:
assortment_probability = {
    "Hypermarket": 0.90,
    "Supermarket": 0.75,
    "Express": 0.45,
    "Neighborhood": 0.35
}

In [128]:
df_store["assortment_probability"] = (
    df_store["store_type"]
    .map(assortment_probability)
)

In [129]:
df_store[
    [
        "store_id",
        "store_type",
        "assortment_probability"
    ]
]

,store_id,store_type,assortment_probability
0,GM001,Hypermarket,0.90
1,GM002,Hypermarket,0.90
2,GM003,Supermarket,0.75
3,GM004,Supermarket,0.75
4,GM005,Supermarket,0.75
5,GM006,Express,0.45
6,GM007,Express,0.45
7,GM008,Hypermarket,0.90
8,GM009,Hypermarket,0.90
9,GM010,Supermarket,0.75


### Store-Level Product Assignment

In [130]:
product_ids = df_product["product_id"].to_numpy()

product_weights = (
    df_product["product_demand_weight"].to_numpy()
)

In [131]:
product_probabilities = (
    product_weights / product_weights.sum()
)

### Create Product Assignment Array

In [132]:
df_sales["product_id"] = None

### Build Store Product Pools

In [133]:
store_product_pools = {}

for _, store in df_store.iterrows():

    store_id = store["store_id"]
    store_type = store["store_type"]

    probability = assortment_probability[store_type]

    mask = np.random.random(len(df_product)) < probability

    eligible_products = df_product.loc[mask]

    # Safety check
    if len(eligible_products) == 0:
        eligible_products = df_product

    store_product_pools[store_id] = eligible_products

### Inspect Store Assortment

In [134]:
store_assortment_summary = pd.DataFrame({
    "store_id": store_product_pools.keys(),
    "assortment_size": [
        len(products)
        for products in store_product_pools.values()
    ]
})

In [135]:
store_assortment_summary

,store_id,assortment_size
0,GM001,452
1,GM002,444
2,GM003,371
3,GM004,386
4,GM005,364
5,GM006,233
6,GM007,210
7,GM008,465
8,GM009,445
9,GM010,379


### Assign Products to Transactions

In [136]:
for store_id in df_sales["store_id"].unique():

    mask = df_sales["store_id"] == store_id
    n = mask.sum()

    product_pool = store_product_pools[store_id]

    weights = product_pool["product_demand_weight"].to_numpy()

    probabilities = weights / weights.sum()

    df_sales.loc[mask, "product_id"] = np.random.choice(
        product_pool["product_id"].to_numpy(),
        size=n,
        p=probabilities
    )

### Validate Product IDs

In [137]:
df_sales["product_id"].isna().sum()

np.int64(0)

In [138]:
df_sales["product_id"].isin(
    df_product["product_id"]
).all()

np.True_

In [139]:
df_sales["product_id"].nunique()

500

### Check Product Distribution

In [140]:
product_sales_distribution = (
    df_sales["product_id"]
    .value_counts()
    .rename_axis("product_id")
    .reset_index(name="transaction_lines")
)

In [141]:
product_sales_distribution.head(20)

,product_id,transaction_lines
0,P0449,884
1,P0308,845
2,P0315,813
3,P0418,807
4,P0406,800
5,P0261,683
6,P0214,680
7,P0090,675
8,P0016,668
9,P0295,665


### Category Distribution

In [142]:
product_category_check = product_sales_distribution.merge(
    df_product[
        [
            "product_id",
            "category"
        ]
    ],
    on="product_id",
    how="left"
)

In [143]:
category_distribution = (
    product_category_check
    .groupby("category")["transaction_lines"]
    .sum()
    .sort_values(ascending=False)
)

category_distribution

category
Food & Beverages     37626
Fresh Food           21997
Health & Wellness    13880
Household            13594
Personal Care        10917
Beauty                9588
Electronics           6682
Baby Care             5716
Name: transaction_lines, dtype: int64

### Bring Product Price Into df_sales

In [144]:
product_price_map = (
    df_product
    .set_index("product_id")["selling_price"]
    .to_dict()
)

In [145]:
df_sales["unit_price_reference"] = (
    df_sales["product_id"]
    .map(product_price_map)
)

In [146]:
df_sales[
    [
        "product_id",
        "unit_price_reference"
    ]
].head()

,product_id,unit_price_reference
0,P0188,119.33
1,P0389,46.52
2,P0128,201.78
3,P0106,93.87
4,P0276,50.19


In [147]:
df_sales["unit_price_reference"].isna().sum()

np.int64(0)

### Generate Quantity

In [148]:
price_median = df_product["selling_price"].median()

In [149]:
quantity_factor = (
    price_median
    / df_sales["unit_price_reference"]
)

In [150]:
quantity_factor = quantity_factor.clip(
    lower=0.5,
    upper=3.0
)

In [151]:
base_quantity = np.random.poisson(
    lam=1.2,
    size=len(df_sales)
) + 1

In [152]:
df_sales["quantity"] = np.round(
    base_quantity * quantity_factor
).astype(int)

In [153]:
df_sales["quantity"] = (
    df_sales["quantity"]
    .clip(lower=1, upper=10)
)

### Inspect Quantity

In [154]:
df_sales["quantity"].describe()

count    120000.000000
mean          3.076542
std           2.375226
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          10.000000
Name: quantity, dtype: float64

In [155]:
df_sales["quantity"].value_counts().sort_index()

quantity
1     36923
2     29047
3     19372
4      8845
5      5240
6      9607
7      2013
8      1036
9      4719
10     3198
Name: count, dtype: int64

### Quantity by Category

In [156]:
quantity_category_check = df_sales.merge(
    df_product[
        [
            "product_id",
            "category"
        ]
    ],
    on="product_id",
    how="left"
)

In [157]:
quantity_category_check.groupby(
    "category"
)["quantity"].mean().sort_values(
    ascending=False
)

category
Food & Beverages     4.011083
Fresh Food           3.545347
Personal Care        3.365851
Household            2.793880
Baby Care            2.428097
Health & Wellness    1.789481
Beauty               1.703484
Electronics          1.571685
Name: quantity, dtype: float64

In [158]:
df_sales.drop(
    columns="unit_price_reference",
    inplace=True
)

In [159]:
df_sales.columns.tolist()

['transaction_id',
 'transaction_date',
 'store_id',
 'customer_id',
 'store_city',
 'product_id',
 'quantity']

In [160]:
df_sales.shape

(120000, 7)

## Critical Validation

### Transaction IDs

In [161]:
df_sales["transaction_id"].is_unique

True

In [162]:
df_sales["store_id"].isin(
    df_store["store_id"]
).all()

np.True_

In [163]:
df_sales["customer_id"].isin(
    df_customer["customer_id"]
).all()

np.True_

In [164]:
df_sales["product_id"].isin(
    df_product["product_id"]
).all()

np.True_

In [165]:
(df_sales["quantity"] > 0).all()

np.True_

In [166]:
df_sales["product_id"].isna().sum()

np.int64(0)

In [167]:
df_sales["quantity"].isna().sum()

np.int64(0)

## Pricing, Gross Sales & Discount Logic

### Inspect Your Product Master

In [168]:
df_product.columns.tolist()

['product_id',
 'product_name',
 'category',
 'subcategory',
 'brand',
 'unit_cost',
 'selling_price',
 'category_weight',
 'product_variation',
 'product_demand_weight']

In [169]:
df_product[
    [
        "product_id",
        "category",
        "selling_price"
    ]
].head(10)

,product_id,category,selling_price
0,P0001,Baby Care,102.30
1,P0002,Beauty,129.53
2,P0003,Personal Care,61.94
3,P0004,Electronics,1009.61
4,P0005,Baby Care,20.29
5,P0006,Food & Beverages,84.06
6,P0007,Fresh Food,80.61
7,P0008,Electronics,460.72
8,P0009,Fresh Food,100.44
9,P0010,Baby Care,131.12


In [170]:
df_product["selling_price"].describe()

count     500.000000
mean      130.277100
std       144.738415
min         3.190000
25%        47.642500
50%        86.510000
75%       163.702500
max      1009.610000
Name: selling_price, dtype: float64

In [171]:
(df_product["selling_price"] > 0).all()

np.True_

### Bring Product Price Into Sales
### Create a loop

In [172]:
product_price_map = (
    df_product
    .set_index("product_id")["selling_price"]
    .to_dict()
)

In [173]:
df_sales["unit_price"] = (
    df_sales["product_id"]
    .map(product_price_map)
)

In [174]:
df_sales[
    [
        "product_id",
        "quantity",
        "unit_price"
    ]
].head(10)

,product_id,quantity,unit_price
0,P0188,1,119.33
1,P0389,2,46.52
2,P0128,1,201.78
3,P0106,4,93.87
4,P0276,2,50.19
5,P0177,1,94.51
6,P0443,1,177.23
7,P0307,10,40.01
8,P0239,1,106.44
9,P0295,4,98.21


### Validate the Price Assignment

In [175]:
df_sales["unit_price"].isna().sum()

np.int64(0)

In [176]:
(df_sales["unit_price"] > 0).all()

np.True_

## Base Price vs Transaction Price

### Add Realistic Price Variation

In [177]:
price_variation = np.random.normal(
    loc=1.0,
    scale=0.02,
    size=len(df_sales)
)

In [178]:
price_variation = np.clip(
    price_variation,
    0.95,
    1.05
)

In [179]:
df_sales["unit_price"] = (
    df_sales["unit_price"]
    * price_variation
).round(2)

### Check Price Distribution

In [180]:
df_sales["unit_price"].describe()

count    120000.000000
mean        114.070692
std         123.417100
min           3.040000
25%          44.700000
50%          76.940000
75%         144.860000
max        1060.090000
Name: unit_price, dtype: float64

In [181]:
df_sales[
    ["product_id", "unit_price"]
].head(20)

,product_id,unit_price
0,P0188,122.75
1,P0389,45.94
2,P0128,206.42
3,P0106,94.45
4,P0276,49.48
5,P0177,96.17
6,P0443,175.92
7,P0307,39.81
8,P0239,106.18
9,P0295,96.04


### Now Design Promotion Logic
### Store-Type Promotion Behavior

In [200]:
store_promo_probability = {
    "Hypermarket": 0.25,
    "Supermarket": 0.30,
    "Express": 0.35,
    "Neighborhood": 0.40
}

In [201]:
df_store["promo_probability"] = (
    df_store["store_type"]
    .map(store_promo_probability)
)

In [202]:
df_store[
    [
        "store_id",
        "store_type",
        "promo_probability"
    ]
]

,store_id,store_type,promo_probability
0,GM001,Hypermarket,0.25
1,GM002,Hypermarket,0.25
2,GM003,Supermarket,0.30
3,GM004,Supermarket,0.30
4,GM005,Supermarket,0.30
5,GM006,Express,0.35
6,GM007,Express,0.35
7,GM008,Hypermarket,0.25
8,GM009,Hypermarket,0.25
9,GM010,Supermarket,0.30


### Map Promotion Probability to Transactions

In [203]:
store_promo_map = (
    df_store
    .set_index("store_id")["promo_probability"]
    .to_dict()
)

In [204]:
df_sales["promo_probability"] = (
    df_sales["store_id"]
    .map(store_promo_map)
)

In [205]:
df_sales["promo_probability"].isna().sum()

np.int64(0)

### Generate Promotion Flag

In [206]:
promotion_random = np.random.random(
    len(df_sales)
)

In [207]:
df_sales["is_promotional"] = (
    promotion_random
    < df_sales["promo_probability"]
)

In [208]:
df_sales["is_promotional"].value_counts()

is_promotional
False    86348
True     33652
Name: count, dtype: int64

In [209]:
df_sales["is_promotional"].mean()

np.float64(0.2804333333333333)

### Generate Discounts

In [210]:
discount_levels = np.array([
    0.05,
    0.10,
    0.15,
    0.20,
    0.25
])

In [211]:
discount_probabilities = np.array([
    0.30,
    0.30,
    0.20,
    0.15,
    0.05
])

### Create Discount Percentage

In [212]:
df_sales["discount_pct"] = 0.0

In [213]:
promo_mask = df_sales["is_promotional"]

In [214]:
promo_mask.sum()

np.int64(33652)

In [215]:
df_sales.loc[promo_mask, "discount_pct"] = (
    np.random.choice(
        discount_levels,
        size=promo_mask.sum(),
        p=discount_probabilities
    )
)

In [216]:
df_sales["discount_pct"].value_counts().sort_index()

discount_pct
0.00    86348
0.05     9951
0.10    10097
0.15     6752
0.20     5112
0.25     1740
Name: count, dtype: int64

### Calculate Gross Sales

In [217]:
df_sales["gross_sales"] = (
    df_sales["quantity"]
    * df_sales["unit_price"]
).round(2)

In [218]:
df_sales[
    [
        "quantity",
        "unit_price",
        "gross_sales"
    ]
].head(10)

,quantity,unit_price,gross_sales
0,1,122.75,122.75
1,2,45.94,91.88
2,1,206.42,206.42
3,4,94.45,377.80
4,2,49.48,98.96
5,1,96.17,96.17
6,1,175.92,175.92
7,10,39.81,398.10
8,1,106.18,106.18
9,4,96.04,384.16


### Validate Gross Sales

In [219]:
gross_sales_check = (
    df_sales["quantity"]
    * df_sales["unit_price"]
).round(2)

In [220]:
np.isclose(
    df_sales["gross_sales"],
    gross_sales_check
).all()

np.True_

### Calculate Discount Amount

In [221]:
df_sales["discount_amount"] = (
    df_sales["gross_sales"]
    * df_sales["discount_pct"]
).round(2)

In [222]:
df_sales[
    [
        "gross_sales",
        "discount_pct",
        "discount_amount"
    ]
].head(10)

,gross_sales,discount_pct,discount_amount
0,122.75,0.00,0.00
1,91.88,0.25,22.97
2,206.42,0.05,10.32
3,377.80,0.00,0.00
4,98.96,0.00,0.00
5,96.17,0.00,0.00
6,175.92,0.00,0.00
7,398.10,0.00,0.00
8,106.18,0.05,5.31
9,384.16,0.00,0.00


### Validate Discount Amount

In [223]:
(
    df_sales["discount_amount"]
    <= df_sales["gross_sales"]
).all()

np.True_

In [224]:
df_sales["discount_amount"].min()

np.float64(0.0)

### Calculate Net Sales

In [225]:
df_sales["net_sales"] = (
    df_sales["gross_sales"]
    - df_sales["discount_amount"]
).round(2)

In [226]:
df_sales[
    [
        "gross_sales",
        "discount_amount",
        "net_sales"
    ]
].head(10)

,gross_sales,discount_amount,net_sales
0,122.75,0.00,122.75
1,91.88,22.97,68.91
2,206.42,10.32,196.10
3,377.80,0.00,377.80
4,98.96,0.00,98.96
5,96.17,0.00,96.17
6,175.92,0.00,175.92
7,398.10,0.00,398.10
8,106.18,5.31,100.87
9,384.16,0.00,384.16


### Validate Net Sales

In [227]:
net_sales_check = (
    df_sales["gross_sales"]
    - df_sales["discount_amount"]
).round(2)

In [228]:
np.isclose(
    df_sales["net_sales"],
    net_sales_check
).all()

np.True_

In [229]:
(
    df_sales["net_sales"] >= 0
).all()

np.True_

### Inspect the Financial Distribution

In [230]:
df_sales[
    [
        "quantity",
        "unit_price",
        "gross_sales",
        "discount_pct",
        "discount_amount",
        "net_sales"
    ]
].describe()

,quantity,unit_price,gross_sales,discount_pct,discount_amount,net_sales
count,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000
mean,3.076542,114.070692,219.888056,0.033145,7.275922,212.612134
std,2.375226,123.417100,176.372250,0.061858,18.308106,171.484755
min,1.000000,3.040000,9.120000,0.000000,0.000000,7.400000
25%,1.000000,44.700000,108.180000,0.000000,0.000000,105.340000
50%,2.000000,76.940000,183.160000,0.000000,0.000000,177.640000
75%,4.000000,144.860000,267.880000,0.050000,6.070000,260.820000
max,10.000000,1060.090000,2927.160000,0.250000,436.160000,2927.160000


### Compare Promotional vs Non-Promotional Sales

In [231]:
df_sales.groupby(
    "is_promotional"
)[
    [
        "gross_sales",
        "discount_amount",
        "net_sales",
        "quantity"
    ]
].mean()

,gross_sales,discount_amount,net_sales,quantity
is_promotional,,,,
False,219.923471,0.000000,219.923471,3.077222
True,219.797186,25.945282,193.851905,3.074795


### Calculate Overall Discount Rate

In [232]:
overall_discount_rate = (
    df_sales["discount_amount"].sum()
    / df_sales["gross_sales"].sum()
)

In [233]:
overall_discount_rate

np.float64(0.03308920891230035)

### Check Discounts by Store

In [234]:
store_discount_summary = (
    df_sales.groupby("store_id")
    .agg(
        gross_sales=("gross_sales", "sum"),
        discount_amount=("discount_amount", "sum"),
        net_sales=("net_sales", "sum")
    )
)

In [235]:
store_discount_summary["discount_pct"] = (
    store_discount_summary["discount_amount"]
    / store_discount_summary["gross_sales"]
)

In [236]:
store_discount_summary.sort_values(
    "discount_pct",
    ascending=False
).head(10)

,gross_sales,discount_amount,net_sales,discount_pct
store_id,,,,
GM022,388378.02,19364.69,369013.33,0.049860
GM024,310733.82,15075.65,295658.17,0.048516
GM020,221963.95,10735.57,211228.38,0.048366
GM015,169300.02,7854.15,161445.87,0.046392
GM025,189845.42,8372.80,181472.62,0.044103
GM019,366021.61,15434.16,350587.45,0.042167
GM012,280403.68,11463.22,268940.46,0.040881
GM006,277020.37,11114.84,265905.53,0.040123
GM007,277945.77,11043.79,266901.98,0.039734


In [237]:
df_sales.drop(
    columns="promo_probability",
    inplace=True
)

### Critical Validation Block

In [238]:
print("Rows:", len(df_sales))

print(
    "Missing product:",
    df_sales["product_id"].isna().sum()
)

print(
    "Missing price:",
    df_sales["unit_price"].isna().sum()
)

print(
    "Invalid quantity:",
    (df_sales["quantity"] <= 0).sum()
)

print(
    "Invalid discount:",
    (
        (df_sales["discount_pct"] < 0)
        |
        (df_sales["discount_pct"] > 0.25)
    ).sum()
)

print(
    "Gross sales validation:",
    np.isclose(
        df_sales["gross_sales"],
        (
            df_sales["quantity"]
            * df_sales["unit_price"]
        ).round(2)
    ).all()
)

print(
    "Net sales validation:",
    np.isclose(
        df_sales["net_sales"],
        (
            df_sales["gross_sales"]
            - df_sales["discount_amount"]
        ).round(2)
    ).all()
)

Rows: 120000
Missing product: 0
Missing price: 0
Invalid quantity: 0
Invalid discount: 0
Gross sales validation: True
Net sales validation: True


### Cost, Gross Profit & Gross Margin

### Inspect the Product Cost

In [239]:
df_product.columns.tolist()

['product_id',
 'product_name',
 'category',
 'subcategory',
 'brand',
 'unit_cost',
 'selling_price']

In [240]:
df_product[
    [
        "product_id",
        "selling_price",
        "unit_cost"
    ]
].head(10)

,product_id,selling_price,unit_cost
0,P0001,102.30,81.18
1,P0002,129.53,93.02
2,P0003,61.94,47.67
3,P0004,1009.61,790.37
4,P0005,20.29,14.90
5,P0006,84.06,69.63
6,P0007,80.61,59.13
7,P0008,460.72,362.12
8,P0009,100.44,73.35
9,P0010,131.12,91.71


In [241]:
df_product["unit_cost"].describe()

count    500.000000
mean      99.034200
std      117.282069
min        2.590000
25%       36.277500
50%       66.585000
75%      120.962500
max      795.430000
Name: unit_cost, dtype: float64

In [242]:
(df_product["unit_cost"] > 0).all()

np.True_

In [243]:
(
    df_product["unit_cost"]
    <
    df_product["selling_price"]
).mean()

np.float64(1.0)

### Bring Unit Cost Into df_sales

In [244]:
product_cost_map = (
    df_product
    .set_index("product_id")["unit_cost"]
    .to_dict()
)

In [245]:
df_sales["unit_cost"] = (
    df_sales["product_id"]
    .map(product_cost_map)
)

In [246]:
df_sales[
    [
        "product_id",
        "quantity",
        "unit_price",
        "unit_cost"
    ]
].head(10)

,product_id,quantity,unit_price,unit_cost
0,P0188,1,122.75,87.36
1,P0389,2,45.94,35.23
2,P0128,1,206.42,159.10
3,P0106,4,94.45,77.09
4,P0276,2,49.48,38.29
5,P0177,1,96.17,74.89
6,P0443,1,175.92,126.65
7,P0307,10,39.81,32.40
8,P0239,1,106.18,77.03
9,P0295,4,96.04,74.31


### Validate Cost Mapping

In [247]:
df_sales["unit_cost"].isna().sum()

np.int64(0)

In [248]:
(df_sales["unit_cost"] > 0).all()

np.True_

In [249]:
(
    df_sales["unit_cost"]
    <
    df_sales["unit_price"]
).mean()

np.float64(1.0)

### Generate Total Product Cost

In [250]:
df_sales["total_cost"] = (
    df_sales["quantity"]
    * df_sales["unit_cost"]
).round(2)

In [251]:
df_sales[
    [
        "quantity",
        "unit_cost",
        "total_cost"
    ]
].head(10)

,quantity,unit_cost,total_cost
0,1,87.36,87.36
1,2,35.23,70.46
2,1,159.10,159.10
3,4,77.09,308.36
4,2,38.29,76.58
5,1,74.89,74.89
6,1,126.65,126.65
7,10,32.40,324.00
8,1,77.03,77.03
9,4,74.31,297.24


### Validate Total Cost

In [252]:
total_cost_check = (
    df_sales["quantity"]
    * df_sales["unit_cost"]
).round(2)

In [253]:
np.isclose(
    df_sales["total_cost"],
    total_cost_check
).all()

np.True_

### Calculate Gross Profit

In [284]:
df_sales["gross_profit"] = (
    df_sales["net_sales"]
    - df_sales["total_cost"]
).round(2)

In [ ]:
df_sales[
    [
        "net_sales",
        "total_cost",
        "gross_profit"
    ]
].head(10)

,net_sales,total_cost,gross_profit
0,122.75,87.36,35.39
1,68.91,70.46,-1.55
2,196.10,159.10,37.00
3,377.80,308.36,69.44
4,98.96,76.58,22.38
5,96.17,74.89,21.28
6,175.92,126.65,49.27
7,398.10,324.00,74.10
8,100.87,77.03,23.84
9,384.16,297.24,86.92


In [287]:
(df_sales["gross_profit"] < 0).sum()

np.int64(2832)

In [288]:
df_sales.loc[
    df_sales["gross_profit"] < 0,
    [
        "product_id",
        "quantity",
        "unit_price",
        "unit_cost",
        "discount_pct",
        "net_sales",
        "total_cost",
        "gross_profit"
    ]
].head(20)

,product_id,quantity,unit_price,unit_cost,discount_pct,net_sales,total_cost,gross_profit
1,P0389,2,45.94,35.23,0.25,68.91,70.46,-1.55
24,P0474,1,63.21,50.72,0.20,50.57,50.72,-0.15
45,P0496,9,39.89,31.03,0.25,269.26,279.27,-10.01
117,P0311,2,116.87,93.88,0.20,186.99,187.76,-0.77
146,P0205,3,61.57,51.39,0.20,147.77,154.17,-6.40
219,P0117,1,64.88,52.80,0.25,48.66,52.80,-4.14
233,P0379,9,11.97,9.88,0.20,86.18,88.92,-2.74
320,P0090,3,16.53,12.89,0.25,37.19,38.67,-1.48
418,P0327,1,161.35,132.31,0.20,129.08,132.31,-3.23
438,P0033,3,65.88,52.94,0.20,158.11,158.82,-0.71


### Calculate Gross Margin %

In [289]:
df_sales["gross_margin_pct"] = np.where(
    df_sales["net_sales"] != 0,
    (
        df_sales["gross_profit"]
        / df_sales["net_sales"]
    ) * 100,
    0
)

In [290]:
df_sales["gross_margin_pct"] = (
    df_sales["gross_margin_pct"]
    .round(2)
)

In [291]:
df_sales[
    [
        "net_sales",
        "total_cost",
        "gross_profit",
        "gross_margin_pct"
    ]
].head(10)

,net_sales,total_cost,gross_profit,gross_margin_pct
0,122.75,87.36,35.39,28.83
1,68.91,70.46,-1.55,-2.25
2,196.10,159.10,37.00,18.87
3,377.80,308.36,69.44,18.38
4,98.96,76.58,22.38,22.62
5,96.17,74.89,21.28,22.13
6,175.92,126.65,49.27,28.01
7,398.10,324.00,74.10,18.61
8,100.87,77.03,23.84,23.63
9,384.16,297.24,86.92,22.63


### Validate Gross Margin

In [292]:
gross_margin_check = np.where(
    df_sales["net_sales"] != 0,
    (
        df_sales["gross_profit"]
        / df_sales["net_sales"]
    ) * 100,
    0
)

In [293]:
np.isclose(
    df_sales["gross_margin_pct"],
    np.round(gross_margin_check, 2)
).all()

np.True_

### Inspect Overall Financial Performance

In [294]:
df_sales[
    [
        "gross_sales",
        "discount_amount",
        "net_sales",
        "total_cost",
        "gross_profit",
        "gross_margin_pct"
    ]
].describe()

,gross_sales,discount_amount,net_sales,total_cost,gross_profit,gross_margin_pct
count,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000
mean,219.888056,7.275922,212.612134,167.164176,45.447959,20.973532
std,176.372250,18.308106,171.484755,141.060280,40.737247,8.504445
min,9.120000,0.000000,7.400000,7.770000,-224.180000,-24.500000
25%,108.180000,0.000000,105.340000,84.710000,18.820000,16.610000
50%,183.160000,0.000000,177.640000,137.960000,36.840000,22.030000
75%,267.880000,6.070000,260.820000,204.960000,59.530000,26.230000
max,2927.160000,436.160000,2927.160000,2612.720000,539.440000,43.800000


### Calculate Company-Level Financial KPIs

In [295]:
company_financials = {
    "gross_sales": df_sales["gross_sales"].sum(),
    "discount_amount": df_sales["discount_amount"].sum(),
    "net_sales": df_sales["net_sales"].sum(),
    "total_cost": df_sales["total_cost"].sum(),
    "gross_profit": df_sales["gross_profit"].sum()
}

In [296]:
company_financials_df = pd.DataFrame(
    [company_financials]
)

In [297]:
company_financials_df["gross_margin_pct"] = (
    company_financials_df["gross_profit"]
    / company_financials_df["net_sales"]
    * 100
).round(2)

In [298]:
company_financials_df

,gross_sales,discount_amount,net_sales,total_cost,gross_profit,gross_margin_pct
0,26386566.76,873110.62,25513456.14,20059701.07,5453755.07,21.38


### Validate the Financial Equation

In [299]:
np.isclose(
    df_sales["gross_sales"]
    - df_sales["discount_amount"],
    df_sales["net_sales"]
).all()

np.True_

In [300]:
np.isclose(
    df_sales["net_sales"]
    - df_sales["total_cost"],
    df_sales["gross_profit"]
).all()

np.True_

### Analyze Margin by Category

In [301]:
category_profitability = df_sales.merge(
    df_product[
        [
            "product_id",
            "category"
        ]
    ],
    on="product_id",
    how="left"
)

In [302]:
category_profitability_summary = (
    category_profitability
    .groupby("category")
    .agg(
        gross_sales=("gross_sales", "sum"),
        discount_amount=("discount_amount", "sum"),
        net_sales=("net_sales", "sum"),
        total_cost=("total_cost", "sum"),
        gross_profit=("gross_profit", "sum")
    )
)

In [303]:
category_profitability_summary["gross_margin_pct"] = (
    category_profitability_summary["gross_profit"]
    /
    category_profitability_summary["net_sales"]
    * 100
).round(2)

In [304]:
category_profitability_summary.sort_values(
    "gross_profit",
    ascending=False
)

,gross_sales,discount_amount,net_sales,total_cost,gross_profit,gross_margin_pct
category,,,,,,
Food & Beverages,6141760.84,203608.53,5938152.31,4895480.55,1042671.76,17.56
Health & Wellness,3506502.92,117203.97,3389298.95,2482638.46,906660.49,26.75
Beauty,2889393.69,95798.70,2793594.99,1901569.31,892025.68,31.93
Fresh Food,3816025.21,126207.22,3689817.99,2918665.95,771152.04,20.90
Electronics,4076668.13,135663.98,3941004.15,3386473.43,554530.72,14.07
Household,2738694.01,89034.41,2649659.60,2111042.90,538616.70,20.33
Personal Care,1990684.17,65738.30,1924945.87,1445549.04,479396.83,24.90
Baby Care,1226837.79,39855.51,1186982.28,918281.43,268700.85,22.64


### Analyze Store Profitability

In [305]:
store_financials = (
    df_sales
    .groupby("store_id")
    .agg(
        gross_sales=("gross_sales", "sum"),
        discount_amount=("discount_amount", "sum"),
        net_sales=("net_sales", "sum"),
        total_cost=("total_cost", "sum"),
        gross_profit=("gross_profit", "sum"),
        transactions=("transaction_id", "nunique"),
        units_sold=("quantity", "sum")
    )
)

In [306]:
store_financials["gross_margin_pct"] = (
    store_financials["gross_profit"]
    /
    store_financials["net_sales"]
    * 100
).round(2)

In [307]:
store_financials["atv"] = (
    store_financials["net_sales"]
    /
    store_financials["transactions"]
).round(2)

In [308]:
store_financials.sort_values(
    "net_sales",
    ascending=False
).head(10)

,gross_sales,discount_amount,net_sales,total_cost,gross_profit,transactions,units_sold,gross_margin_pct,atv
store_id,,,,,,,,,
GM008,3125338.22,89988.14,3035350.08,2382281.87,653068.21,14227,43899,21.52,213.35
GM009,2935064.49,85041.45,2850023.04,2233331.22,616691.82,13223,41120,21.64,215.54
GM013,2864893.51,82555.02,2782338.49,2182657.71,599680.78,12949,40863,21.55,214.87
GM002,2756442.43,82311.11,2674131.32,2090445.21,583686.11,12816,39486,21.83,208.66
GM001,2139320.18,63845.87,2075474.31,1625556.46,449917.85,9801,30758,21.68,211.76
GM023,1249929.71,44032.12,1205897.59,945256.51,260641.08,5620,16522,21.61,214.57
GM014,1250300.36,45837.18,1204463.18,948641.23,255821.95,5698,17398,21.24,211.38
GM011,1207238.97,41834.36,1165404.61,915456.38,249948.23,5462,16545,21.45,213.37
GM003,939027.36,33408.20,905619.16,716045.82,189573.34,4325,13388,20.93,209.39


### Find High-Sales / Low-Margin Stores

In [309]:
store_financials.sort_values(
    "net_sales",
    ascending=False
).head(10)

,gross_sales,discount_amount,net_sales,total_cost,gross_profit,transactions,units_sold,gross_margin_pct,atv
store_id,,,,,,,,,
GM008,3125338.22,89988.14,3035350.08,2382281.87,653068.21,14227,43899,21.52,213.35
GM009,2935064.49,85041.45,2850023.04,2233331.22,616691.82,13223,41120,21.64,215.54
GM013,2864893.51,82555.02,2782338.49,2182657.71,599680.78,12949,40863,21.55,214.87
GM002,2756442.43,82311.11,2674131.32,2090445.21,583686.11,12816,39486,21.83,208.66
GM001,2139320.18,63845.87,2075474.31,1625556.46,449917.85,9801,30758,21.68,211.76
GM023,1249929.71,44032.12,1205897.59,945256.51,260641.08,5620,16522,21.61,214.57
GM014,1250300.36,45837.18,1204463.18,948641.23,255821.95,5698,17398,21.24,211.38
GM011,1207238.97,41834.36,1165404.61,915456.38,249948.23,5462,16545,21.45,213.37
GM003,939027.36,33408.20,905619.16,716045.82,189573.34,4325,13388,20.93,209.39


In [310]:
store_financials.sort_values(
    "gross_margin_pct"
).head(10)

,gross_sales,discount_amount,net_sales,total_cost,gross_profit,transactions,units_sold,gross_margin_pct,atv
store_id,,,,,,,,,
GM006,277020.37,11114.84,265905.53,215123.51,50782.02,1208,3689,19.10,220.12
GM024,310733.82,15075.65,295658.17,236559.78,59098.39,1344,4198,19.99,219.98
GM007,277945.77,11043.79,266901.98,212198.15,54703.83,1212,3543,20.50,220.22
GM017,219507.75,8318.08,211189.67,167766.07,43423.60,992,3016,20.56,212.89
GM012,280403.68,11463.22,268940.46,213517.24,55423.22,1240,3690,20.61,216.89
GM018,861658.79,30864.14,830794.65,659097.71,171696.94,3829,11618,20.67,216.97
GM019,366021.61,15434.16,350587.45,277757.90,72829.55,1570,4524,20.77,223.30
GM003,939027.36,33408.20,905619.16,716045.82,189573.34,4325,13388,20.93,209.39
GM005,769246.26,26667.50,742578.76,587063.03,155515.73,3577,11294,20.94,207.60


In [311]:
df_sales.columns.tolist()

['transaction_id',
 'transaction_date',
 'store_id',
 'customer_id',
 'store_city',
 'product_id',
 'quantity',
 'unit_price',
 'is_promotional',
 'discount_pct',
 'gross_sales',
 'discount_amount',
 'net_sales',
 'unit_cost',
 'total_cost',
 'gross_profit',
 'store_type',
 'gross_margin_pct']

In [312]:
df_sales.shape

(120000, 18)

### Final Financial Validation Block

In [327]:
print("Rows:", len(df_sales))

print(
    "Missing product:",
    df_sales["product_id"].isna().sum()
)

print(
    "Missing unit price:",
    df_sales["unit_price"].isna().sum()
)

print(
    "Missing unit cost:",
    df_sales["unit_cost"].isna().sum()
)

print(
    "Invalid quantity:",
    (df_sales["quantity"] <= 0).sum()
)

print(
    "Negative gross sales:",
    (df_sales["gross_sales"] < 0).sum()
)

print(
    "Negative discount:",
    (df_sales["discount_amount"] < 0).sum()
)

print(
    "Net sales validation:",
    np.isclose(
        df_sales["gross_sales"]
        - df_sales["discount_amount"],
        df_sales["net_sales"]
    ).all()
)

print(
    "Cost validation:",
    np.isclose(
        df_sales["quantity"]
        * df_sales["unit_cost"],
        df_sales["total_cost"]
    ).all()
)

print(
    "Gross profit validation:",
    np.isclose(
        df_sales["net_sales"]
        - df_sales["total_cost"],
        df_sales["gross_profit"]
    ).all()
)

print(
    "Gross margin validation:",
    np.isclose(
        df_sales["gross_profit"]
        / df_sales["net_sales"] * 100,
        df_sales["gross_margin_pct"]
    ).all()
)

Rows: 120000
Missing product: 0
Missing unit price: 0
Missing unit cost: 0
Invalid quantity: 0
Negative gross sales: 0
Negative discount: 0
Net sales validation: True
Cost validation: True
Gross profit validation: True
Gross margin validation: False
